# Readmission Risk Prediction — End-to-End ML Pipeline

**INSY 662 · Enterprise Data Science · Group Project**

This notebook orchestrates the full pipeline using modular Python scripts in `src/`.
Each phase maps to a specific course module (see README.md for the full mapping).

**Key Design Decision:** The train/test split (Phase 5) is performed **before** feature selection (Phase 4) to prevent data leakage. PCA and Mutual Information are fit on training data only.

---

In [ ]:
# ═══════════════════════════════════════════
# SETUP
# ═══════════════════════════════════════════
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

from src.config import OUTPUT_DIR, DATA_FILE
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("All imports loaded successfully.")

---
## Phases 1–2: Data Loading & Cleaning
*Module II: Enterprise DS Lifecycle, Best Practices*

In [ ]:
from src.data_cleaning import run_cleaning

df = run_cleaning()  # Uses config.DATA_FILE by default

print(f"
Quick missing values check:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({"missing": missing[missing > 0], "pct": missing_pct[missing > 0]}))

---
## Phase 3: Feature Engineering
*Module III: Feature Engineering (ICD-9 grouping, medication aggregation, derived features)*

In [ ]:
from src.feature_engineering import run_feature_engineering

df = run_feature_engineering(df)
print(f"
Encoded DataFrame shape: {df.shape}")

---
## Phase 5: Train/Test Split (Patient-Level Grouped)
*Module II: Veridical Data Science*

**IMPORTANT:** The split happens BEFORE feature selection to prevent data leakage.

In [ ]:
from src.splitting import separate_X_y, train_test_split_grouped

X, y, patient_ids = separate_X_y(df)
X_train, X_test, y_train, y_test, groups_train = train_test_split_grouped(X, y, patient_ids)

---
## Phase 4: Feature Selection & Dimensionality Reduction (Post-Split)
*Module III: Mutual Information, PCA, zero-variance removal*

**LEAKAGE FIX:** PCA and MI are computed on training data only.

In [ ]:
from src.feature_selection import run_feature_selection

X_train, X_test, mi_df, feature_names = run_feature_selection(
    X_train, X_test, y_train, output_dir=OUTPUT_DIR
)

---
## Phase 6: Baseline Models
*Module IV: Model Evaluation*

In [ ]:
from src.modeling import heuristic_baseline, logistic_baseline

heuristic_score = heuristic_baseline(X_test, y_test)
lr_baseline, lr_scaler, lr_probs = logistic_baseline(X_train, X_test, y_train, y_test)

---
## Phase 7: SOTA Supervised Learning (5-Fold CV)
*Module IV: SOTA Classification — LR, RF, XGBoost, LightGBM*

In [ ]:
from src.modeling import run_cv_all_models

cv_results, scale_pos = run_cv_all_models(X_train, y_train, groups_train)

---
## Phase 8: Final Model Training & Test Evaluation
*Module IV: Ensemble Learning, Stacking*

In [ ]:
from src.modeling import train_final_models, compare_models

models, scaler = train_final_models(X_train, X_test, y_train, y_test, scale_pos)
results_df, best_name = compare_models(models, heuristic_score, y_test)

best_model, best_probs = models[best_name]

---
## Phase 9: ROC & Precision-Recall Curves
*Module IV: Model Evaluation*

In [ ]:
from src.evaluation import plot_roc_pr_curves

model_probs = {name: probs for name, (_, probs) in models.items()}
plot_roc_pr_curves(model_probs, y_test,
                   save_path=os.path.join(OUTPUT_DIR, "roc_pr_curves.png"))

---
## Phase 10: Probability Calibration
*Module II: Best Practices*

In [ ]:
from src.calibration import calibrate_model

cal_model = calibrate_model(
    best_model, best_probs, X_test, y_test, best_name,
    save_path=os.path.join(OUTPUT_DIR, "calibration.png"),
)

---
## Phase 11: Explainability (SHAP)
*Module I: AI Strategy, Responsible AI*

In [ ]:
from src.explainability import compute_shap_values, shap_summary_plots, shap_waterfall_examples

explainer, shap_values, X_shap_sample = compute_shap_values(
    best_model, X_test, feature_names
)
shap_summary_plots(shap_values, X_shap_sample, save_dir=OUTPUT_DIR)

In [ ]:
# Per-patient waterfall examples
shap_waterfall_examples(explainer, X_test, best_probs, feature_names)

---
## Phase 12: Fairness Audit
*Module I: Responsible AI*

In [ ]:
from src.fairness import audit_fairness_by_race

fairness_df = audit_fairness_by_race(
    X_test, y_test, best_probs,
    save_path=os.path.join(OUTPUT_DIR, "fairness_audit.png"),
)

---
## Phase 13: Operational Outputs (Risk Tiers + Top-K Outreach)
*Module I: Data as a Business*

In [ ]:
from src.operational import generate_outreach_list

outreach_df = generate_outreach_list(
    explainer, X_test, best_probs, feature_names,
    save_path=os.path.join(OUTPUT_DIR, "outreach_list.csv"),
)

---
## Phase 14: Confusion Matrix & Classification Report
*Module IV: Model Evaluation*

In [ ]:
from src.evaluation import plot_confusion_matrix

plot_confusion_matrix(y_test, best_probs, best_name,
                      save_path=os.path.join(OUTPUT_DIR, "confusion_matrix.png"))

---
## Phase 15: Cumulative Gains & Lift Chart
*Module II: Enterprise DS*

In [ ]:
from src.evaluation import plot_gains_and_lift

plot_gains_and_lift(y_test, best_probs, best_name,
                    save_path=os.path.join(OUTPUT_DIR, "gains_lift.png"))

---
## Final Summary

In [ ]:
print("=" * 70)
print("  FINAL SUMMARY")
print("=" * 70)

print(f"""
PROJECT: 30-Day Readmission Risk Prediction for Diabetic Patients
DATASET: UCI Diabetes 130-US Hospitals | {len(df):,} encounters
TARGET:  Binary (readmitted <30 days) | {y.mean()*100:.1f}% positive rate
SPLIT:   Patient-level grouped | Train: {len(X_train):,} | Test: {len(X_test):,}
BEST MODEL: {best_name}
""")

print(results_df.to_string(index=False))

print("""

COURSE CONCEPTS APPLIED:
  Module I:   AI Strategy, Responsible AI (SHAP, Fairness Audit)
  Module II:  End-to-end DS lifecycle, best practices, leakage prevention
  Module III: Feature engineering (ICD grouping, derived features)
              Feature selection (MI, PCA) — fitted on training data only
  Module IV:  SOTA Classification (LR, RF, XGBoost, LightGBM, Stacking)
              Model evaluation (ROC-AUC, PR-AUC, Brier, top-K)
  Module V:   PCA for exploration (unsupervised dimensionality reduction)
  Module VII: Presentation-ready outputs and visualisations
""")
print("PIPELINE COMPLETE ✓")